In [1]:
import json 
from src.llm import LLM
from src.prompt import *
from src.utils import iterate_schelling_data, coordination_index

prompt_schema = Level0  # prompting technique

/home/emanuele/focal-points/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
# Get the data
with open('./data/schelling.jsonl', 'r') as file:
    data = json.load(file)

# Get the combinations of problems
problems = iterate_schelling_data(data)

In [3]:
# model = lambda x: random.choice(['<answer>A</answer>', '<answer>B</answer>'])  # Mock model for demonstration
trials = 10  # Number of trials for each problem
model = LLM(model_id="meta-llama/Llama-3.2-1B",
            num_return_sequences=3)

# Run the experiments
responses = {}
for idx in problems:
    responses[idx] = {}
    for problem in problems[idx]:
        if problem not in responses[idx]:
            responses[idx][problem] = []
            
        prompt = Level0.prefix + problem + Level0.suffix
        for _ in range(trials):
            response = model.generate_batch(prompt)
            responses[idx][problem].append(response)
        

Device set to use cuda:0
/home/emanuele/focal-points/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [4]:
logs = []
for idx in responses:
    for problem in responses[idx]:
        log = {
            'idx': idx,
            'prompt': problem,
            "responses": responses[idx][problem]
        }
    
        logs.append(log)
    
with open('./logs/schelling_responses.jsonl', 'w') as f:
    json.dump(logs, f, indent=2)

In [5]:
# Clean the data 
for idx in responses:
    for response in responses[idx]:
        try:
            responses[idx][response] = [r.replace('<answer>', '').replace('</answer>', '').strip() for r in responses[idx][response]]
        except:
            # Delete the response if it fails to clean
            del responses[idx][response]

RuntimeError: dictionary changed size during iteration

In [ ]:
# Compute the coordination index
results = []
for idx in responses:
    for response in responses[idx]:
        if responses[idx][response]:
            c_index, nc_index = coordination_index(responses[idx][response])
            
            result = {
                "idx": idx,
                "problem": response,
                "coordination_index": c_index,
                "normalised_coordination_index": nc_index
            }
            
            results.append(result)
              
# Write the results to a file
with open("./results/schelling_results.jsonl", "a") as f:
    json.dump(results, f, indent=2)
    